# `add_embeddings` smoke walkthrough

End-to-end demo of `synth_setter.pipeline.data.add_embeddings`:

1. Build a tiny in-memory **Lance** dataset shaped like a `generate_dataset` /
   `finalize_dataset` shard (an `audio` tensor column plus `mel_spec` /
   `param_array`, with `ShardMetadata` embedded in the schema).
2. Run the **m2l + CLAP** add step. It appends:
   - `clap` — a `FixedSizeList<float32, 512>` LAION-CLAP embedding, **vector-
     searchable** (Lance builds an IVF_PQ index on it for large datasets).
   - `m2l` — a `fixed_shape_tensor<float32, (C*D, T)>` music2latent latent
     (retrievable, not a search vector).
3. Read the dataset back into a pandas `DataFrame` holding **just the params and
   the two new embedding columns**.
4. Run a **vector search** over the `clap` column with `nearest=`.

> Loading the real encoders downloads the music2latent and
> `laion/clap-htsat-unfused` checkpoints, so step 2 needs `music2latent`,
> `transformers`, and `torchaudio` installed and will pull weights on first run.

In [ ]:
import tempfile
from pathlib import Path

import lance
import numpy as np
import pandas as pd

from synth_setter.data.vst.shapes import (
    AUDIO_FIELD,
    DATASET_FIELD_DTYPES,
    MEL_SPEC_FIELD,
    PARAM_ARRAY_FIELD,
    audio_dataset_shape,
    mel_dataset_shape,
    param_array_dataset_shape,
)
from synth_setter.pipeline.data.lance_shard import (
    lance_schema,
    read_shard_metadata,
    record_batch_from_arrays,
    write_lance_dataset,
)
from synth_setter.pipeline.schemas.shard_metadata import ShardMetadata

## 1. Build a smoke Lance dataset

Four rows of 1 s, 44.1 kHz stereo noise — the same on-disk layout a real shard
carries, including the `ShardMetadata` the add step reads the sample rate from.

In [ ]:
rows, channels, sample_rate, duration, num_params = 4, 2, 44_100, 1.0, 56
shapes = {
    AUDIO_FIELD: audio_dataset_shape(rows, channels, sample_rate, duration),
    MEL_SPEC_FIELD: mel_dataset_shape(rows, channels, sample_rate, duration),
    PARAM_ARRAY_FIELD: param_array_dataset_shape(rows, num_params),
}
metadata = ShardMetadata(
    velocity=100,
    signal_duration_seconds=duration,
    sample_rate=sample_rate,
    channels=channels,
    min_loudness=-55.0,
)
schema = lance_schema(shapes, metadata)

rng = np.random.default_rng(0)
arrays = {
    AUDIO_FIELD: (0.1 * rng.standard_normal(shapes[AUDIO_FIELD])).astype(
        DATASET_FIELD_DTYPES[AUDIO_FIELD]
    ),
    MEL_SPEC_FIELD: np.zeros(shapes[MEL_SPEC_FIELD], DATASET_FIELD_DTYPES[MEL_SPEC_FIELD]),
    PARAM_ARRAY_FIELD: rng.random((rows, num_params)).astype(
        DATASET_FIELD_DTYPES[PARAM_ARRAY_FIELD]
    ),
}

uri = str(Path(tempfile.mkdtemp()) / "smoke.lance")
write_lance_dataset(uri, schema, [record_batch_from_arrays(arrays, schema)])
lance.dataset(uri).schema.names

## 2. Run the m2l + CLAP add step

`add_embeddings` reads only the `audio` column per batch and appends the `m2l`
tensor and `clap` vector columns. The sample rate comes from the shard metadata,
exactly as the `python -m synth_setter.pipeline.data.add_embeddings <uri>` CLI
does it.

We pass `build_index=False` here because IVF_PQ needs ~256 rows to train; on a
real (large) dataset keep the default so `clap` gets an ANN index. Exact
`nearest` search (step 4) works either way.

In [ ]:
from synth_setter.pipeline.data.add_embeddings import (
    CLAP_FIELD,
    M2L_FIELD,
    add_embeddings,
    load_clap_audio_encoder,
    load_m2l_audio_encoder,
)

dataset = lance.dataset(uri)
sample_rate = int(read_shard_metadata(dataset.schema).sample_rate)

# First call downloads the music2latent + CLAP checkpoints.
m2l_encode = load_m2l_audio_encoder()
clap_encode = load_clap_audio_encoder()

add_embeddings(dataset, m2l_encode, clap_encode, sample_rate, build_index=False)
lance.dataset(uri).schema.to_string()

## 3. DataFrame of params + new embeddings

Project only `param_array`, `m2l`, and `clap`. `clap` rows come back as
fixed-size lists; `m2l` as a fixed-shape tensor.

In [ ]:
table = lance.dataset(uri).to_table(columns=[PARAM_ARRAY_FIELD, M2L_FIELD, CLAP_FIELD])

frame = pd.DataFrame(
    {
        PARAM_ARRAY_FIELD: list(
            table.column(PARAM_ARRAY_FIELD).combine_chunks().to_numpy_ndarray()
        ),
        M2L_FIELD: list(table.column(M2L_FIELD).combine_chunks().to_numpy_ndarray()),
        CLAP_FIELD: [
            np.asarray(row, dtype=np.float32) for row in table.column(CLAP_FIELD).to_pylist()
        ],
    }
)
frame["m2l_shape"] = frame[M2L_FIELD].map(np.shape)
frame["clap_shape"] = frame[CLAP_FIELD].map(np.shape)
frame[["m2l_shape", "clap_shape"]]

## 4. Vector search over `clap`

Query the `clap` column with `nearest=`. Using row 0's own embedding as the
query, it should come back first (distance ~0). On a dataset with an IVF_PQ
index this is ANN; here (4 rows, no index) Lance falls back to an exact scan —
same API either way.

In [ ]:
query = np.asarray(frame[CLAP_FIELD].iloc[0], dtype=np.float32)
neighbors = lance.dataset(uri).to_table(
    columns=[PARAM_ARRAY_FIELD],
    nearest={"column": CLAP_FIELD, "q": query, "k": 3},
)
neighbors.select(["_distance"]).to_pandas()